# SYP PARTS9 → Drive raw (original-style)

Same behavior as the old local `PART9S_to_drive_raw.ipynb`:
`chdir` into `01_raw` and `to_csv` with relative filenames.

Prefer the BAT (runs this notebook via nbconvert), which matches the
scheduler setup that historically worked on SYP.


In [ ]:
from sqlalchemy import create_engine
import pandas as pd
import os

from dotenv import load_dotenv
load_dotenv()

server = os.getenv("PARTS9_SYP_SERVER", os.getenv("KSS_PC_SERVER", "KSS-PC"))
database = os.getenv("PARTS9_SYP_DATABASE", "PARTS9")

connection_string = (
    "mssql+pyodbc://@"
    f"{server}/"
    f"{database}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

engine = create_engine(connection_string)


In [ ]:
def read_last_5y(table):
    query = f"""
    SELECT *
    FROM dbo.{table}
    WHERE BILLDATE >= DATEADD(YEAR, -5, (SELECT MAX(BILLDATE) FROM dbo.{table}))
    """
    return pd.read_sql(query, engine)

df_sidet = read_last_5y("SIDET")
df_pidet = read_last_5y("PIDET")
df_simas = read_last_5y("SIMAS")
df_pimas = read_last_5y("PIMAS")


In [ ]:
df_icmas = pd.read_sql(
    "SELECT * FROM dbo.ICMAS",
    engine,
)


In [ ]:
import os
from src.kcw import paths

# Prefer configured Drive root; fall back to the classic Shared Drive path.
try:
    raw_dir = paths.raw_dir()
except FileNotFoundError:
    raw_dir = r"G:\Shared drives\KCW-Data\kcw_analytics\01_raw"

os.chdir(raw_dir)
print("cwd=", os.getcwd())

# Same relative to_csv order as the original local notebook.
df_pidet.to_csv("raw_syp_pidet_purchase_lines.csv", index=False, encoding="utf-8-sig")
df_pimas.to_csv("raw_syp_pimas_purchase_bills.csv", index=False, encoding="utf-8-sig")
df_sidet.to_csv("raw_syp_sidet_sales_lines.csv", index=False, encoding="utf-8-sig")
df_simas.to_csv("raw_syp_simas_sales_bills.csv", index=False, encoding="utf-8-sig")
df_icmas.to_csv("raw_syp_icmas_products.csv", index=False, encoding="utf-8-sig")

print("DONE rows", {
    "pidet": len(df_pidet),
    "pimas": len(df_pimas),
    "sidet": len(df_sidet),
    "simas": len(df_simas),
    "icmas": len(df_icmas),
})
